In [1]:
import asyncio
import json
import logging
from logging import INFO
import os
from datetime import datetime, timezone, timedelta

from dotenv import load_dotenv

from graphiti_core import Graphiti
from graphiti_core.nodes import EpisodeType
from graphiti_core.search.search_config_recipes import NODE_HYBRID_SEARCH_RRF

In [2]:
logging.basicConfig(
    level=INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
logger = logging.getLogger(__name__)

load_dotenv()

neo4j_uri = os.environ.get('NEO4J_URI')
neo4j_user = os.environ.get('NEO4J_USERNAME')
neo4j_password = os.environ.get('NEO4J_PASSWORD')

if not neo4j_uri or not neo4j_user or not neo4j_password:
    raise ValueError('NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD must be set')

In [ ]:
graphiti = Graphiti(neo4j_uri, neo4j_user, neo4j_password)

# await graphiti.build_indices_and_constraints()

2025-12-28 09:26:29 - neo4j.notifications - INFO - Received notification from DBMS server: {severity: INFORMATION} {code: Neo.ClientNotification.Schema.IndexOrConstraintAlreadyExists} {category: SCHEMA} {title: `CREATE RANGE INDEX community_uuid IF NOT EXISTS FOR (e:Community) ON (e.uuid)` has no effect.} {description: `RANGE INDEX community_uuid FOR (e:Community) ON (e.uuid)` already exists.} {position: None} for query: 'CREATE INDEX community_uuid IF NOT EXISTS FOR (n:Community) ON (n.uuid)'
2025-12-28 09:26:29 - neo4j.notifications - INFO - Received notification from DBMS server: {severity: INFORMATION} {code: Neo.ClientNotification.Schema.IndexOrConstraintAlreadyExists} {category: SCHEMA} {title: `CREATE RANGE INDEX entity_group_id IF NOT EXISTS FOR (e:Entity) ON (e.group_id)` has no effect.} {description: `RANGE INDEX entity_group_id FOR (e:Entity) ON (e.group_id)` already exists.} {position: None} for query: 'CREATE INDEX entity_group_id IF NOT EXISTS FOR (n:Entity) ON (n.group_i

In [10]:
episodes = [
    {
        'name': 'OpenAI',
        'content': {
            'name': 'OpenAI',
            'founded': '2015-12-8',
            'founders': 'Sam Altman',
            'type': 'AI 연구 개발 기업',
            'official_website': 'https://openai.com/'
        },
        'type': EpisodeType.json,
        'description': 'article metadata',
    },
    {
        'name': 'OpenAI의 기본 모델',
        'content': '2024년 5월 기준, OpenAI의 기본 모델은 GPT-4o입니다.',
        'type': EpisodeType.text,
        'description': 'news article',
    }
]

print(f'총 {len(episodes)}개의 에피소드가 정의되었습니다.')

총 2개의 에피소드가 정의되었습니다.


In [12]:
for i, episode in enumerate(episodes):
    await graphiti.add_episode(
        name=episode['name'],
        episode_body=episode['content']
        if isinstance(episode['content'], str)
        else json.dumps(episode['content']),
        source=episode['type'],
        source_description=episode['description'],
        reference_time=datetime.now(timezone.utc),
    )
    print(f"Added episode: {episode['name']} ({episode['type'].value})")

2025-12-27 15:30:05 - neo4j.notifications - WARNING - Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: entity_edges)} {position: line: 17, column: 7, offset: 473} for query: '\n                                    MATCH (e:Episodic)\n                                    WHERE e.valid_at <= $reference_time\n                                    \nAND e.group_id IN $group_ids\nAND e.source = $source\n        RETURN\n        \n    e.uuid AS uuid,\n    e.name AS name,\n    e.group_id AS group_id,\n    e.created_at AS created_at,\n    e.source AS source,\n    e.source_description AS source_description,\n    e.

Added episode: OpenAI (json)


2025-12-27 15:30:30 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2025-12-27 15:30:30 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 15:30:31 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 15:30:32 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2025-12-27 15:30:36 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2025-12-27 15:30:36 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 15:30:38 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 15:30:38 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 15:30:39 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2025-

Added episode: OpenAI의 기본 모델 (text)


In [14]:
episodes = [
    {
        'name': '새로운 모델 출시',
        'content': 'OpenAI의 현재 기본 모델은 8월 8일부로 GPT-5로 변경되었습니다.'
        'GPT-5는 전반적으로 훨씬 더 스마트하며, 특히 수학, 코딩, 시각적 인식, 의료 분야의 학술적 및 인간 평가 벤치마크에서의 성과에서 확인할 수 있습니다.'
        '수학(도구 없이 AIME 2025에서 94.6%), 실제 코딩(SWEBench에서 74.9%, 에이더 플리글롯에서 88%), 멀티모달 이해(MMU에서 84.2%), 의료(HealthBench Hard에서 46.2%) 전반에서 새롭게 최고 기록을 세웠으며 이러한 이점은 일상 사용에도 적용됩니다.',
        'type': EpisodeType.text,
        'description': 'news article',
    }
]

print(f"총 {len(episodes)} 개의 에피소드가 정의되었습니다.")

총 1 개의 에피소드가 정의되었습니다.


In [15]:
for i, episode in enumerate(episodes):
    await graphiti.add_episode(
        name=episode['name'],
        episode_body = episode['content']
        if isinstance(episode['content'], str)
        else json.dumps(episode['content']),
        source=episode['type'],
        source_description=episode['description'],
        reference_time=datetime.now(timezone.utc),
    )
    print(f'Added episode: {episode["name"]} ({episode["type"].value})')

2025-12-27 19:12:50 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2025-12-27 19:12:51 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 19:12:51 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 19:12:51 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 19:12:52 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 19:12:52 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 19:12:52 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 19:12:52 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-12-27 19:12:57 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
202

Added episode: 새로운 모델 출시 (text)


In [4]:
query = 'OpenAI의 정보를 알려주세요.'
print(f'검색 중: "{query}"')
results = await graphiti.search(query)

print('\n검색 결과:')
for result in results:
    print(f'UUID: {result.uuid}')
    print(f'Fact: {result.fact}')
    if hasattr(result, 'valid_at') and result.valid_at:
        print(f'Valid from: {result.valid_at}')
    if hasattr(result, 'invalid_at') and result.invalid_at:
        print(f'Valid until: {result.invalid_at}')
    print('---')

검색 중: "OpenAI의 정보를 알려주세요."


2025-12-28 09:28:46 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



검색 결과:
UUID: a36f4fa8-b0c0-43af-bf1c-7c3bcbdf88cc
Fact: OpenAI is an AI 연구 개발 기업
Valid from: 2025-12-27 06:30:05+00:00
---
UUID: 78c979fb-e39c-4e40-822a-fc122a9cb500
Fact: OpenAI's official website is https://openai.com/
Valid from: 2025-12-27 06:30:05+00:00
---
UUID: 8a6dec10-3e6b-44bd-9176-59eeee941be8
Fact: OpenAI was founded by Sam Altman
Valid from: 2015-12-08 00:00:00+00:00
---
UUID: d1d9a8ad-6c1e-4ed6-8b8a-209be67e526f
Fact: As of May 2024, OpenAI's default model is GPT-4o.
Valid from: 2024-05-01 00:00:00+00:00
Valid until: 2025-08-08 00:00:00+00:00
---
UUID: 6468aa7c-7122-441f-a60d-ea24a03e7ae2
Fact: As of August 8, 2025, GPT-5 became the current default model of OpenAI.
Valid from: 2025-08-08 00:00:00+00:00
---
UUID: e3886294-bf54-4397-844f-d87808c94ed2
Fact: GPT-5 achieved the highest record in practical coding on 에이더 플리글롯 with a score of 88%.
---


In [5]:
query = 'OpenAI의 정보를 알려주세요.'

center_node_uuid = results[0].source_node_uuid

print('\n그래프 거리를 기반으로 검색 결과 재정렬:')
print(f'중심 노드 UUID 사용: {center_node_uuid}')

reranked_results = await graphiti.search(
    query, center_node_uuid=center_node_uuid
)

print('\n재정렬된 검색 결과:')
for result in reranked_results:
    print(f'UUID: {result.uuid}')
    print(f'Fact: {result.fact}')
    if hasattr(result, 'valid_at') and result.valid_at:
        print(f'Valid from: {result.valid_at}')
    if hasattr(result, 'invalid_at') and result.invalid_at:
        print(f'Valid until: {result.invalid_at}')
    print('---')


그래프 거리를 기반으로 검색 결과 재정렬:
중심 노드 UUID 사용: 7f85c485-91a1-47a1-918c-7cbc932e2174


2025-12-28 10:15:14 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



재정렬된 검색 결과:
UUID: a36f4fa8-b0c0-43af-bf1c-7c3bcbdf88cc
Fact: OpenAI is an AI 연구 개발 기업
Valid from: 2025-12-27 06:30:05+00:00
---
UUID: 78c979fb-e39c-4e40-822a-fc122a9cb500
Fact: OpenAI's official website is https://openai.com/
Valid from: 2025-12-27 06:30:05+00:00
---
UUID: 8a6dec10-3e6b-44bd-9176-59eeee941be8
Fact: OpenAI was founded by Sam Altman
Valid from: 2015-12-08 00:00:00+00:00
---
UUID: d1d9a8ad-6c1e-4ed6-8b8a-209be67e526f
Fact: As of May 2024, OpenAI's default model is GPT-4o.
Valid from: 2024-05-01 00:00:00+00:00
Valid until: 2025-08-08 00:00:00+00:00
---
UUID: 6468aa7c-7122-441f-a60d-ea24a03e7ae2
Fact: As of August 8, 2025, GPT-5 became the current default model of OpenAI.
Valid from: 2025-08-08 00:00:00+00:00
---
UUID: e3886294-bf54-4397-844f-d87808c94ed2
Fact: GPT-5 achieved the highest record in practical coding on 에이더 플리글롯 with a score of 88%.
---


In [ ]:
node_search_config = NODE_HYBRID_SEARCH_RRF.model_copy(deep=True)
node_search_config.limit = 5

node_search_results = await graphiti._search(
    query='gpt-5',
    config=node_search_config,
)

print('\n노드 검색 결과:')
for node in node_search_results.nodes:
    print(f'Node UUID: {node.uuid}')
    print(f'Node Name: {node.name}')
    node_summary = node.summary[:100] + '...' if len(node.summary) > 100 else node.summary
    print(f'Content Summary: {node_summary}')
    print(f'Node Labels: {", ".join(node.labels)}')
    print(f'Created At: {node.created_at}')
    if hasattr(node, 'attributes') and node.attributes:
        